#### setup

In [ ]:
import json
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

from library.data_utils import *
from library.models import *
from library.training import *

In [ ]:
# device and seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(seed=0)

In [ ]:
# parameters
dataset = 'synsum'
train_frac = 0.8

#### data

In [ ]:
# load data and split
df = pd.read_csv(f"./data/datasets/{dataset}.csv", index_col=0)
train_df, val_df, test_df = make_splits(df=df, train_frac=train_frac, seed=0)

In [ ]:
# set confounders
confounders_text = ['dysp', 'cough', 'pain', 'nasal', 'fever_none', 'fever_low', 'fever_high']
confounders_full = confounders_text + ['self_empl', 'asthma', 'smoking', 'COPD', 'winter','hay_fever']

#### train nuisance models

In [ ]:
# set parameters
input_dim = len(confounders_full)
params = dict(
    hidden_dim=64,
    learning_rate=5e-4,
    weight_decay=0,
    batch_size=128,
    max_epochs=50,
    patience=5)

In [ ]:
# init data loaders
train_loader, val_loader = make_nuisance_loaders(train_df, val_df, confounders_full, params["batch_size"])

# train propensity model
propensity_model = ClassificationHead(input_dim=input_dim, hidden_dim=params["hidden_dim"]).to(device)
propensity_model, info = train_propensity(propensity_model, train_loader, val_loader, device, lr=params["learning_rate"], 
                                          weight_decay=params["weight_decay"], max_epochs=params["max_epochs"], patience=params["patience"], seed=0)
# store
torch.save(propensity_model.state_dict(), './data/checkpoints/propensity_model.pt')

In [ ]:
# init data loaders (idem for T = 1)
train_resp = train_df[train_df["T"] == 0]
val_resp = val_df[val_df["T"] == 0]
train_loader, val_loader = make_nuisance_loaders(train_resp, val_resp, confounders_full, params["batch_size"])

# train response model
response_model_control = RegressionHead(input_dim=input_dim, hidden_dim=params["hidden_dim"]).to(device)
response_model_control, info = train_response(response_model_control, train_loader, val_loader, device,lr=params["learning_rate"], weight_decay=params["weight_decay"], 
                                              max_epochs=params["max_epochs"], patience=params["patience"], seed=0)

# store
torch.save(response_model_control.state_dict(), './data/checkpoints/response_model_control.pt')